##Semantic Segmentation of Pets

In [1]:
# ## Objectives
#Build a **reliable semantic segmentation pipeline** in PyTorch
#Perform **meaningful EDA** to understand dataset challenges
#Establish a strong **baseline model**
#Improve performance using **transfer learning**
#Evaluate models using **IoU and Dice metrics**
#Visually analyze **successes and failure cases**

# ###Problem Definition
#**Foreground (pet)**, **Background**
# ###Dataset Characteristics
#High-resolution RGB images
#Pixel-level annotations provided as trimaps
#Significant variation in object size and appearance
#Ambiguous boundaries, especially around fur
# ## Annotation Format
# The provided pixel annotations use the following encoding:
# | Label | Meaning |
# |------|--------|
# | 1 | Foreground (pet) |
# | 2 | Background |
# | 3 | Not classified / boundary region|

In [2]:
!pip install -q segmentation-models-pytorch
import warnings
warnings.filterwarnings('ignore')
import os
import numpy as np
import pandas as pd
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

SEED = 42
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device: ', DEVICE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.0 MB/s eta 0:00:00
Using device:  cuda


## Dataset class

In [3]:
class OxfordPetDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transforms=None, file_list=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transforms = transforms

        if file_list is not None:
            self.image_filenames = file_list
        else:
            self.image_filenames = sorted([
                f for f in os.listdir(images_dir) if f.endswith(".jpg")
            ])

    def __len__(self):
        return len(self.image_filenames)

    def _load_mask(self, mask_path):
        mask = np.array(Image.open(mask_path))

        # 1 = foreground, 2 = background, 3 = not classified
        binary_mask = np.where(mask == 2, 0, 1).astype(np.float32)
        return binary_mask

    def __getitem__(self, idx):
        img_name = self.image_filenames[idx]

        image_path = os.path.join(self.images_dir, img_name)
        mask_path = os.path.join(
            self.masks_dir,
            "trimaps", # Added 'trimaps' subdirectory
            img_name.replace(".jpg", ".png")
        )

        image = Image.open(image_path).convert("RGB")
        mask = self._load_mask(mask_path)

        if self.transforms:
            augmented = self.transforms(image=np.array(image), mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        if not torch.is_tensor(mask):
            mask = torch.from_numpy(mask)

        mask = (mask > 0.5).float()
        return image, mask.unsqueeze(0)

In [4]:
def visualize_sample(dataset, idx):
    image, mask = dataset[idx]

    if isinstance(image, torch.Tensor):
        image = image.permute(1, 2, 0).cpu().numpy()
    else:
        image = np.array(image)

    if isinstance(mask, torch.Tensor):
        mask = mask.squeeze().cpu().numpy()
    else:
        mask = np.squeeze(mask)

    fig, axes  = plt.subplots(1, 3, figsize = (15, 5))
    axes[0].imshow(image)
    axes[0].set_title('Image')
    axes[0].axis('off')

    axes[1].imshow(mask, cmap = 'gray')
    axes[1].set_title('Mask')
    axes[1].axis('off')

    axes[2].imshow(image)
    axes[2].imshow(mask, alpha = 0.5, cmap = 'jet')
    axes[2].set_title('Overlay')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

# Exploratory Data Analysis (EDA)

In [5]:
# Cell 1: Install dependencies
!pip install datasets huggingface_hub timm albumentations -q

import os, random, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import albumentations as A
from albumentations.pytorch import ToTensorV2
from datasets import load_dataset
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Using device: cuda


### Prepare dataset for EDA

In [6]:
import os, requests, tarfile
from pathlib import Path

MASK_DIR = Path("oxford_pet_masks")

def download_masks():
    url = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz"
    tar_path = "annotations.tar.gz"

    if not MASK_DIR.exists():
        print("Downloading annotations (masks).")
        r = requests.get(url, stream=True)
        with open(tar_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Extracting.")
        with tarfile.open(tar_path, 'r:gz') as tar:
            tar.extractall(".")
        os.rename("annotations", str(MASK_DIR))
        os.remove(tar_path)
        print("Done!")
    else:
        print("Masks already downloaded.")

download_masks()

trimap_dir = MASK_DIR / "trimaps"
print(f"Trimap directory exists: {trimap_dir.exists()}")
print(f"Sample files: {list(trimap_dir.iterdir())[:5]}")

def convert_mask_binary(mask_input) -> np.ndarray:

    mask = np.array(mask_input)
    binary = np.zeros_like(mask, dtype=np.uint8)
    binary[mask == 1] = 1   # foreground  → 1
    binary[mask == 2] = 0   # background  → 0
    binary[mask == 3] = 1   # boundary    → 1 (project requirement)
    return binary

#Define mask loader
from PIL import Image as PILImage

def load_mask_by_id(image_id: str) -> np.ndarray:
    mask_path = trimap_dir / f"{image_id}.png"
    if not mask_path.exists():
        raise FileNotFoundError(f"Mask not found:{mask_path}")
    return np.array(PILImage.open(mask_path))

Extracting.
Done!
Trimap directory exists: True
Sample files: [PosixPath('oxford_pet_masks/trimaps/wheaten_terrier_144.png'), PosixPath('oxford_pet_masks/trimaps/._saint_bernard_16.png'), PosixPath('oxford_pet_masks/trimaps/pomeranian_91.png'), PosixPath('oxford_pet_masks/trimaps/german_shorthaired_46.png'), PosixPath('oxford_pet_masks/trimaps/american_bulldog_99.png')]


In [7]:
# Download original images to ensure filename compatibility with masks
import os, requests, tarfile
from pathlib import Path

ACTUAL_IMAGE_DIR = Path("oxford_pet_images")

def download_images():
    url = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz"
    tar_path = "images.tar.gz"
    if not ACTUAL_IMAGE_DIR.exists():
        print("Downloading images...")
        r = requests.get(url, stream=True)
        with open(tar_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Extracting...")
        with tarfile.open(tar_path, 'r:gz') as tar:
            tar.extractall(".")
        os.rename("images", str(ACTUAL_IMAGE_DIR))
        os.remove(tar_path)
        print("Done!")

download_images()

IMAGE_DIR = str(ACTUAL_IMAGE_DIR)
MASK_DIR = "oxford_pet_masks"
TRIMAP_DIR = os.path.join(MASK_DIR, "trimaps")

all_image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(".jpg")]
existing_files = []
for f in all_image_files:
    mask_name = f.replace(".jpg", ".png")
    if os.path.exists(os.path.join(TRIMAP_DIR, mask_name)):
        existing_files.append(f)

existing_files.sort()

dataset_eda = OxfordPetDataset(
    images_dir=IMAGE_DIR,
    masks_dir=MASK_DIR,
    transforms=None,
    file_list=existing_files
)

print(f"Using IMAGE_DIR: {IMAGE_DIR}")
print(f"Dataset initialized with {len(dataset_eda)} samples.")

Extracting...
Done!
Using IMAGE_DIR: oxford_pet_images
Dataset initialized with 7390 samples.


In [ ]:
# ## Foreground Pixel Distribution
# A key aspect of segmentation datasets is the proportion of foreground pixels relative to the background.
# We compute the **foreground pixel ratio** for each image:
# This helps identify class imbalance
# Reveals samples with missing or invalid annotations
# Informs loss function and evaluation metric choices

from tqdm.auto import tqdm

def compute_mask_coverage(dataset):
    coverages = []
    for _, mask in tqdm(dataset, desc="Computing coverage"):
        fg_pixels = mask.sum()
        total_pixels = mask.numel()
        coverages.append((fg_pixels / total_pixels).item())

    return np.array(coverages)

coverages = compute_mask_coverage(dataset_eda)

plt.figure(figsize = (8, 4))
plt.hist(coverages, bins = 30, color = 'royalblue')
plt.title('Foreground Pixel Ratio Distribution')
plt.xlabel("Foreground ratio")
plt.ylabel("Number of images")
plt.show()

print(f'Mean foreground ratio: {coverages.mean():.3f}')
print(f'Min foreground ratio: {coverages.min():.3f}')
print(f'Max foreground ratio: {coverages.max():.3f}')

In [ ]:
dataset = OxfordPetDataset(
    images_dir=IMAGE_DIR,
    masks_dir=MASK_DIR,
    transforms=None
)

print(f"Dataset instantiated with {len(dataset)} samples.")
# Visualize a sample to confirm
visualize_sample(dataset, 0)

Observations
- The ratio is Normally distributed, slightly right-skewed.
- The average foreground ratio is approximately **0.4**
- Some images contain **no foreground pixels at all**
- A very small number of samples are almost entirely foreground
Images with zero foreground pixels provide no learning signal for a foreground segmentation task and are treated as invalid samples.

### Image Resolution Analysis

In [ ]:
widths, heights = [], []

for img,_ in dataset_eda:
    w, h = img.size
    widths.append(w)
    heights.append(h)

plt.figure(figsize = (10, 4))

plt.subplot(1,2,1)
plt.hist(widths, bins = 30, color = 'orange')
plt.title('Image Width Distribution')

plt.subplot(1, 2, 2)
plt.hist(heights, bins = 30, color = 'purple')
plt.title("Image Height Distribution")

plt.tight_layout()
plt.show()

#The dataset exhibits large variation in image resolution, with a small number of high-resolution outliers.
#The majority of images are tightly clustered around **400–500 pixels** in both dimensions.

## Visual Inspection of Samples

In [ ]:
for idx in np.random.choice(range(5000), 5, replace = False):
    visualize_sample(dataset_eda, idx)

## Smaller foreground ratio samples

In [ ]:

small_pet_indices = np.argsort(coverages)[:5]

for idx in small_pet_indices:
    visualize_sample(dataset_eda, idx)

## Larger foreground ratio samples

In [ ]:

large_pet_indices = np.argsort(coverages)[-5:]

for idx in large_pet_indices:
    visualize_sample(dataset_eda, idx)

# Data Cleaning and Preprocessing
### Data Quality Filtering


## Final dataset after cleaning

In [14]:
#Cleaning Rule Applied
# Only images with a foreground pixel ratio within a valid range are retained for training.
# This filtering step improves label quality while preserving the majority of the dataset.

# Foreground ratio < 5%   -> corrupted / useless mask
# Foreground ratio > 90%  -> too little signal

In [15]:
MIN_FG_RATIO = 0.05
MAX_FG_RATIO = 0.90

valid_indices = np.where(
    (coverages > MIN_FG_RATIO) & (coverages < MAX_FG_RATIO)
)[0]

clean_files = [dataset_eda.image_filenames[i] for i in valid_indices]

print(f"Original samples: {len(dataset_eda)}")
print(f"Clean samples:    {len(clean_files)}")
print(f"\nRemoved {len(dataset_eda) - len(clean_files)} empty-mask/complete-mask samples")

Original samples: 7390
Clean samples:    7309

Removed 81 empty-mask/complete-mask samples


In [16]:
cleaned_dataset_eda = OxfordPetDataset(
    images_dir=IMAGE_DIR,
    masks_dir=MASK_DIR,
    transforms=None,
    file_list = clean_files
)

In [ ]:
cleaned_coverages = compute_mask_coverage(cleaned_dataset_eda)

plt.figure(figsize = (8, 4))
plt.hist(cleaned_coverages, bins = 30, color = 'royalblue')
plt.title('Foreground Pixel Ratio Distribution')
plt.xlabel("Foreground ratio")
plt.ylabel("Number of images")
plt.show()

print(f'Mean foreground ratio: {cleaned_coverages.mean():.3f}')
print(f'Min foreground ration: {cleaned_coverages.min():.3f}')
print(f'Max foreground ration: {cleaned_coverages.max():.3f}')

### Smaller foreground ratio sample (after cleaning)

In [ ]:
small_pet_indices_cleaned = np.argsort(cleaned_coverages)[:5]

for idx in small_pet_indices_cleaned:
    visualize_sample(cleaned_dataset_eda, idx)

### Larger foreground ratio sample (after cleaning)

In [ ]:

large_pet_indices_cleaned = np.argsort(cleaned_coverages)[-5:]

for idx in large_pet_indices_cleaned:
    visualize_sample(cleaned_dataset_eda, idx)

## Train-Validation Split


In [20]:

# Dataset is **cleaned first** (empty / invalid masks removed)
# 80-20 train-validation split
train_files, val_files = train_test_split(
    clean_files,
    test_size=0.2,
    random_state=SEED,
    shuffle=True
)
print(f"Train samples: {len(train_files)}")
print(f"Val samples:   {len(val_files)}")

Train samples: 5847
Val samples:   1462


## Transforms


In [21]:
#into 256 x 256 pixels
train_transforms = A.Compose(
    [
        A.Resize(256, 256),
        A.HorizontalFlip(p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ]
)

val_transforms = A.Compose(
    [
        A.Resize(256, 256),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ]
)

## Dataset Construction


In [22]:
train_dataset = OxfordPetDataset(
    images_dir=IMAGE_DIR,
    masks_dir=MASK_DIR,
    transforms=train_transforms,
    file_list=train_files
)

val_dataset = OxfordPetDataset(
    images_dir=IMAGE_DIR,
    masks_dir=MASK_DIR,
    transforms=val_transforms,
    file_list=val_files
)

# Baseline U-Net (From Scratch)

In [23]:
class DoubleConv(nn.Module):

    #(Conv -> ReLU)*2

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size = 3, padding = 1),
            nn.ReLU(inplace = True),
            nn.Conv2d(out_channels, out_channels, kernel_size = 3, padding = 1),
            nn.ReLU(inplace = True),
        )

    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_channels = 3, out_channels= 1):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size = 2, stride = 2)
        self.dec4 = DoubleConv(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size = 2, stride = 2)
        self.dec3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size = 2, stride = 2)
        self.dec2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size = 2, stride = 2)
        self.dec1 = DoubleConv(128, 64)

        # Output
        self.final = nn.Conv2d(64, out_channels, kernel_size = 1)

    def forward(self, x):

        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        # Bottleneck
        b = self.bottleneck(self.pool(e4))

        # Decoder
        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim = 1))
        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim = 1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim = 1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim = 1))
        return self.final(d1)

In [24]:
model = UNet().to(DEVICE)

# Quick test
x = torch.randn(1, 3, 256, 256).to(DEVICE)
y = model(x)

print("Input shape :", x.shape)
print("Output shape:", y.shape)

Input shape : torch.Size([1, 3, 256, 256])
Output shape: torch.Size([1, 1, 256, 256])


## Loss Function
The baseline model is trained using a **combined loss**:
- **Binary Cross-Entropy with Logits**
- **Dice Loss**


In [25]:
def dice_loss(pred, target, smooth = 1.0):
    pred = torch.sigmoid(pred)
    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()
    dice = (2.0 * intersection + smooth) / (
        pred.sum() + target.sum() + smooth
    )

    return 1 - dice

bce_loss = nn.BCEWithLogitsLoss()

def combined_loss(pred, target):
    return bce_loss(pred, target) + dice_loss(pred, target)

## Evaluation Metrics

Model performance is evaluated using:
- **Intersection over Union (IoU)** - primary metric
- **Dice coefficient** - overlap sensitivity

In [26]:
def iou_score(pred, target, threshold = 0.5, smooth = 1e-6):
    pred = torch.sigmoid(pred) > threshold
    target = target > 0.5

    intersection = (pred & target).float().sum((1,2,3))
    union = (pred | target).float().sum((1,2,3))

    return ((intersection + smooth) / (union + smooth)).mean().item()

def dice_score(pred, target, threshold = 0.5, smooth = 1e-6):
    pred = torch.sigmoid(pred) > threshold
    target = target > 0.5

    intersection = (pred & target).float().sum((1,2,3))
    dice = (2 * intersection + smooth) / (
        pred.float().sum((1,2,3)) +
        target.float().sum((1,2,3)) +
        smooth
    )

    return dice.mean().item()

## Training Setup

- Optimizer: **Adam** Fixed input resolution: 256×256
- Validation metrics computed over the **entire validation set**
- Best model checkpoint selected based on **validation IoU**

In [30]:

optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)

BATCH_SIZE = 16
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = NUM_WORKERS,
    pin_memory = True
)

val_loader = DataLoader(
    val_dataset,
    batch_size = BATCH_SIZE,
    shuffle = False,
    num_workers = NUM_WORKERS,
    pin_memory = True
)

In [31]:
def train_one_epoch(model, loader, optimizer, criterion, device):

    model.train()

    running_loss = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss

@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    iou_scores = []
    dice_scores = []

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        loss = criterion(outputs, masks)

        running_loss += loss.item() * images.size(0)

        iou_scores.append(iou_score(outputs, masks))
        dice_scores.append(dice_score(outputs, masks))

    epoch_loss = running_loss / len(loader.dataset)
    mean_iou = sum(iou_scores) / len(iou_scores)
    mean_dice = sum(dice_scores) / len(dice_scores)

    return epoch_loss, mean_iou, mean_dice

In [ ]:
NUM_EPOCHS = 20

best_val_iou = 0.0
history = {
    'train_loss': [],
    'val_loss': [],
    'val_iou': [],
    'val_dice': []
}

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        combined_loss,
        DEVICE
    )

    val_loss, val_iou, val_dice = validate_one_epoch(
        model,
        val_loader,
        combined_loss,
        DEVICE
    )

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_iou)
    history['val_dice'].append(val_dice)

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"IoU: {val_iou:.4f} | "
        f"Dice: {val_dice:.4f}"
    )

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), "best_unet_baseline.pth")
        print('Best model saved')

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

plt.figure(figsize = (14, 4))

plt.subplot(1, 3, 1)
plt.plot(epochs, history['train_loss'], label = 'Train Loss')
plt.plot(epochs, history['val_loss'], label = 'Val Loss')

plt.xlabel('(Epochs)')
plt.ylabel('(Loss)')
plt.title('Loss Curve')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs, history['val_iou'])

plt.xlabel('(Epochs)')
plt.ylabel('(IoU)')
plt.title('Validation IoU')

plt.subplot(1, 3, 3)
plt.plot(epochs, history['val_dice'])

plt.xlabel('(Epochs)')
plt.ylabel('(Dice)')
plt.title('Validation Dice')

plt.suptitle('Baseline model')
plt.tight_layout()
plt.show()

### Reload the best saved model

In [ ]:
model.load_state_dict(torch.load('best_unet_baseline.pth'))
model.eval()

UNet(
  (enc1): DoubleConv(
    (block): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
    )
  )
  (enc2): DoubleConv(
    (block): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
    )
  )
  (enc3): DoubleConv(
    (block): Sequential(
      (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
    )
  )
  (enc4): DoubleConv(
    (block): Sequential(
      (0): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(5

In [ ]:
@torch.no_grad()
def visualize_prediction(dataset, idx, model, device):
    image, mask = dataset[idx]

    input_tensor = image.unsqueeze(0).to(device)
    pred = model(input_tensor)

    pred_mask = (torch.sigmoid(pred) > 0.5).float()

    # Denormalize image for visualization
    image_np = denormalize(image)

    gt_mask = mask.squeeze().cpu().numpy()
    pred_mask = pred_mask.squeeze().cpu().numpy()

    fig, axes = plt.subplots(1, 4, figsize = (15, 5))

    axes[0].imshow(image_np)
    axes[0].set_title('Image')
    axes[0].axis('off')

    axes[1].imshow(gt_mask, cmap = 'gray')
    axes[1].set_title('Ground Truth')
    axes[1].axis('off')

    axes[2].imshow(pred_mask, alpha = 0.9, cmap = 'jet')
    axes[2].set_title('prediction mask')
    axes[2].axis('off')

    axes[3].imshow(image_np)
    axes[3].imshow(pred_mask, alpha = 0.6, cmap = 'jet')
    axes[3].set_title('prediction on image')
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()


IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225])

def denormalize(tensor):

    if tensor.ndim == 4:
        tensor = tensor[0]

    mean = IMAGENET_MEAN[:, None, None]
    std = IMAGENET_STD[:, None, None]

    tensor = tensor.cpu() * std + mean
    tensor = tensor.clamp(0, 1)

    return tensor.permute(1, 2, 0).numpy()

## Results on random samples

In [ ]:
for idx in np.random.choice(len(val_dataset), 3, replace = False):
    visualize_prediction(val_dataset, idx, model, DEVICE)

## Worst Performance on some images

In [ ]:
@torch.no_grad()
def compute_per_sample_iou(model, dataset, device):
    model.eval()
    scores = []

    for idx in range(len(dataset)):
        image, mask = dataset[idx]

        image = image.unsqueeze(0).to(device)
        mask  = mask.unsqueeze(0).to(device)

        pred = model(image)
        iou = iou_score(pred, mask)

        scores.append((idx, iou))

    return scores


val_ious = compute_per_sample_iou(model, val_dataset, DEVICE)
val_ious = sorted(val_ious, key=lambda x: x[1])

worst_cases = val_ious[:5]

for idx, score in worst_cases:
    print(f"IoU: {score:.3f}")
    visualize_prediction(val_dataset, idx, model, DEVICE)

# Transfer Learning: U-Net with ResNet34
- Stronger low-level edge and texture features, Better boundary representation,Faster and more stable convergence

In [ ]:
resnet_model = smp.Unet(
    encoder_name = 'resnet34',
    encoder_weights = 'imagenet',
    in_channels = 3,
    classes = 1,
    activation = None
).to(DEVICE)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

## Training Strategy
Warm-Up Phase (Frozen Encoder)
- Encoder weights are frozen, Only the decoder is trained
- Allows task-specific adaptation without disrupting pretrained features
- Fine-Tuning Phase (Unfrozen Encoder)


In [ ]:
# Freeze encoder
for param in resnet_model.encoder.parameters():
    param.requires_grad = False

# optimizer
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, resnet_model.parameters()),
    lr = 1e-3
)

# Creterion
criterion = combined_loss


# Train (short warm-up phase)
print('WarmUP phase for few epochs with frozen encoder\n')
WARMUP_EPOCHS = 5

best_val_iou = 0.0
tl_history = {
    'train_loss': [],
    'val_loss': [],
    'val_iou': [],
    'val_dice': []
}

for epoch in range(WARMUP_EPOCHS):
    train_loss = train_one_epoch(
        resnet_model,
        train_loader,
        optimizer,
        combined_loss,
        DEVICE
    )

    val_loss, val_iou, val_dice = validate_one_epoch(
        resnet_model,
        val_loader,
        combined_loss,
        DEVICE
    )

    tl_history['train_loss'].append(train_loss)
    tl_history['val_loss'].append(val_loss)
    tl_history['val_iou'].append(val_iou)
    tl_history['val_dice'].append(val_dice)

    print(
        f"Epoch [{epoch + 1}/{WARMUP_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"IoU: {val_iou:.4f} | "
        f"Dice: {val_dice:.4f}"
    )


# Unfreeze encoder (fine - tuning phase)
print('\nFine tune phase for few more epochs with unfreeze encoder\n')
FINETUNE_EPOCHS = 10

for param in resnet_model.encoder.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(resnet_model.parameters(), lr = 1e-4)

for epoch in range(FINETUNE_EPOCHS):
    train_loss = train_one_epoch(
        resnet_model,
        train_loader,
        optimizer,
        combined_loss,
        DEVICE
    )

    val_loss, val_iou, val_dice = validate_one_epoch(
        resnet_model,
        val_loader,
        combined_loss,
        DEVICE
    )

    tl_history['train_loss'].append(train_loss)
    tl_history['val_loss'].append(val_loss)
    tl_history['val_iou'].append(val_iou)
    tl_history['val_dice'].append(val_dice)

    print(
        f"Epoch [{epoch + 1}/{FINETUNE_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"IoU: {val_iou:.4f} | "
        f"Dice: {val_dice:.4f}"
    )

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(resnet_model.state_dict(), "best_unet_resnet.pth")
        print('Best model saved')

WarmUP phase for few epochs with frozen encoder

Epoch [1/5] | Train Loss: 0.2548 | Val Loss: 0.1867 | IoU: 0.8932 | Dice: 0.9420
Epoch [2/5] | Train Loss: 0.1858 | Val Loss: 0.1711 | IoU: 0.9009 | Dice: 0.9465
Epoch [3/5] | Train Loss: 0.1704 | Val Loss: 0.1629 | IoU: 0.9046 | Dice: 0.9487
Epoch [4/5] | Train Loss: 0.1644 | Val Loss: 0.1568 | IoU: 0.9069 | Dice: 0.9499
Epoch [5/5] | Train Loss: 0.1570 | Val Loss: 0.1554 | IoU: 0.9088 | Dice: 0.9509

Fine tune phase for few more epochs with unfreeze encoder

Epoch [1/10] | Train Loss: 0.1650 | Val Loss: 0.1678 | IoU: 0.9035 | Dice: 0.9478
Best model saved
Epoch [2/10] | Train Loss: 0.1457 | Val Loss: 0.1577 | IoU: 0.9079 | Dice: 0.9503
Best model saved
Epoch [3/10] | Train Loss: 0.1404 | Val Loss: 0.1648 | IoU: 0.9064 | Dice: 0.9489
Epoch [4/10] | Train Loss: 0.1322 | Val Loss: 0.1536 | IoU: 0.9110 | Dice: 0.9517
Best model saved
Epoch [5/10] | Train Loss: 0.1271 | Val Loss: 0.1450 | IoU: 0.9133 | Dice: 0.9532
Best model saved
Epoch [6

## Training Plots

In [ ]:
epochs = range(WARMUP_EPOCHS + FINETUNE_EPOCHS)

plt.figure(figsize = (14, 4))

plt.subplot(1, 3, 1)
plt.plot(epochs, tl_history['train_loss'], label = 'Train Loss')
plt.plot(epochs, tl_history['val_loss'], label = 'Val Loss')
plt.xticks(np.arange(0,WARMUP_EPOCHS + FINETUNE_EPOCHS), np.arange(1, WARMUP_EPOCHS + FINETUNE_EPOCHS + 1))
plt.xlabel('(Epochs)')
plt.ylabel('(Loss)')
plt.title('Loss Curve')
plt.legend()
plt.axvspan(0,WARMUP_EPOCHS, ymin = 0,
            ymax = 1,
            color ='yellow',
            alpha = 0.4)
plt.axvspan(WARMUP_EPOCHS, WARMUP_EPOCHS + FINETUNE_EPOCHS - 1, ymin = 0,
            ymax = 1,
            color ='green',
            alpha = 0.4)

plt.subplot(1, 3, 2)
plt.plot(epochs, tl_history['val_iou'])
plt.xticks(np.arange(0,WARMUP_EPOCHS + FINETUNE_EPOCHS), np.arange(1, WARMUP_EPOCHS + FINETUNE_EPOCHS + 1))
plt.xlabel('(Epochs)')
plt.ylabel('(IoU)')
plt.title('Validation IoU')
plt.axvspan(0,WARMUP_EPOCHS, ymin = 0,
            ymax = 1,
            color ='yellow',
            alpha = 0.4)
plt.axvspan(WARMUP_EPOCHS, WARMUP_EPOCHS + FINETUNE_EPOCHS - 1, ymin = 0,
            ymax = 1,
            color ='green',
            alpha = 0.4)

plt.subplot(1, 3, 3)
plt.plot(epochs, tl_history['val_dice'])
plt.xticks(np.arange(0,WARMUP_EPOCHS + FINETUNE_EPOCHS), np.arange(1, WARMUP_EPOCHS + FINETUNE_EPOCHS + 1))
plt.xlabel('(Epochs)')
plt.ylabel('(Dice)')
plt.title('Validation Dice')
plt.axvspan(0,WARMUP_EPOCHS, ymin = 0,
            ymax = 1,
            color ='yellow',
            alpha = 0.4)
plt.axvspan(WARMUP_EPOCHS, WARMUP_EPOCHS + FINETUNE_EPOCHS - 1, ymin = 0,
            ymax = 1,
            color ='green',
            alpha = 0.4)

plt.suptitle('ResNet-34 model')
plt.tight_layout()
plt.show()

### Reload the best saved model

In [ ]:
resnet_model.load_state_dict(torch.load("best_unet_resnet.pth"))
resnet_model.eval()

Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

In [ ]:
for idx in np.random.choice(len(val_dataset), 3, replace = False):
    visualize_prediction(val_dataset, idx, resnet_model, DEVICE)

## Worst performance on some images

In [ ]:
tl_val_ious = compute_per_sample_iou(resnet_model, val_dataset, DEVICE)
tl_val_ious = sorted(tl_val_ious, key=lambda x: x[1])

tl_worst_cases = tl_val_ious[:5]

for idx, score in tl_worst_cases:
    print(f"IoU: {score:.3f}")
    visualize_prediction(val_dataset, idx, resnet_model, DEVICE)

### Results Summary
The transfer learning model shows:
- Faster convergence compared to the scratch baseline
- Consistent improvement in both **IoU** and **Dice**
- Noticeably sharper boundaries in qualitative results
- Better handling of thin fur and partial occlusion


# Baseline VS Transfer learning (comparison)

## Quantitative comparison

In [ ]:

@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    ious, dices = [], []

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)

        ious.append(iou_score(preds, masks))
        dices.append(dice_score(preds, masks))

    return {
        'IoU': sum(ious) / len(ious),
        'Dice': sum(dices) / len(dices)
    }

baseline_metrics = evaluate_model(model, val_loader, DEVICE)
transfer_metrics = evaluate_model(resnet_model, val_loader, DEVICE)

df = pd.DataFrame([
    {'Model': 'Baseline U-Net', **baseline_metrics},
    {'Model': "Transfer U-Net (ResNet34)", **transfer_metrics}
])

df

,Model,IoU,Dice
0,Baseline U-Net,0.843418,0.910189
1,Transfer U-Net (ResNet34),0.914819,0.954337


Both models are evaluated on the **same validation set**, using identical preprocessing and evaluation metrics to ensure a fair comparison.
The transfer-learning model achieves a **substantial improvement** in both overlap-based metrics.

## Qualitative Comparison
Samples on which the baseline model was strugling

In [ ]:
@torch.no_grad()
def visualize_comparison(dataset, idx, baseline_model, transfer_model, device):
    image, mask = dataset[idx]

    input_tensor = image.unsqueeze(0).to(device)

    pred_base = baseline_model(input_tensor)
    pred_tl = transfer_model(input_tensor)

    pred_base = (torch.sigmoid(pred_base) > 0.5).float()
    pred_tl   = (torch.sigmoid(pred_tl) > 0.5).float()

    image_np = denormalize(image)
    gt_mask = mask.squeeze().cpu().numpy()
    base_mask = pred_base.squeeze().cpu().numpy()
    tl_mask = pred_tl.squeeze().cpu().numpy()

    fig, axes = plt.subplots(1, 4, figsize = (18, 5))

    axes[0].imshow(image_np)
    axes[0].set_title('Image')
    axes[0].axis('off')

    axes[1].imshow(image_np)
    axes[1].imshow(base_mask, alpha = 0.6, cmap = 'jet')
    axes[1].set_title('Baseline U-Net')
    axes[1].axis('off')

    axes[2].imshow(image_np)
    axes[2].imshow(tl_mask, alpha = 0.6, cmap = 'jet')
    axes[2].set_title('Transfer U-Net')
    axes[2].axis('off')

    axes[3].imshow(gt_mask, cmap = 'gray')
    axes[3].set_title('Ground Truth')
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

val_ious_base = compute_per_sample_iou(model, val_dataset, DEVICE)
val_ious_base = sorted(val_ious_base, key=lambda x: x[1])

hard_cases = [idx for idx, _ in val_ious_base[:5]]

for idx in hard_cases:
    visualize_comparison(val_dataset, idx, model, resnet_model, DEVICE)